In [6]:
from hana_ml import dataframe
url, port, user, pwd = "810070ba-df1a-4553-9a82-23690dc0158e.hana.demo-hc-3-haas-hc-dev.dev-aws.hanacloud.ondemand.com", \
443, "PALDEVUSER", "Abcd1234"
cc = dataframe.ConnectionContext(url, port, user, pwd)

In [20]:
import numpy as np
import pandas as pd
np.random.seed(2023)
data = pd.concat((pd.DataFrame(dict(ID=range(128))),
                  pd.DataFrame(np.random.rand(128), columns=['X1'])),
                  axis=1)
#data = pd.DataFrame(np.random.rand(128,2), columns=["X1", 'X2'])

In [21]:
from hana_ml.dataframe import create_dataframe_from_pandas
sim_df = create_dataframe_from_pandas(cc, data,
                                      "FFT_SIM_DATA_TBL",
                                      force=True)

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.72it/s]


In [22]:
from hana_ai.tools.hana_ml_tools.fft_tools import FFT
fft_tool = FFT(cc)

In [28]:
tool_input = dict(input_table='FFT_SIM_DATA_TBL', num_type='imag', inverse=True)
fft_tool.run(tool_input=tool_input)

'{"fft_result_table": "FFT_SIM_DATA_TBL_FFT_RESULT"}'

In [29]:
cc.table("FFT_SIM_DATA_TBL_FFT_RESULT").collect()

,ID,REAL,IMAG
0,0,0.000000,0.485623
1,1,0.011528,-0.026314
2,2,0.030249,-0.021834
3,3,0.002924,0.001773
4,4,-0.021183,0.042836
...,...,...,...
123,123,0.021655,-0.002071
124,124,0.021183,0.042836
125,125,-0.002924,0.001773
126,126,-0.030249,-0.021834


In [25]:
from langchain.agents import initialize_agent, AgentType
from gen_ai_hub.proxy.langchain import init_llm
llm = init_llm('gpt-4o', temperature=0.0, max_tokens=2000) # used to do logical reasoning
tools = [fft_tool] # Add any tools here
agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)

C:\Users\I326292\AppData\Local\Temp\ipykernel_396\1827641142.py:5: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)


In [32]:
instruction_str = "I want to calculate the inverse FFT of the data contained in table FFT_SIM_DATA_TBL, where number type of column X1 is imaginary."
agent_chain.invoke(instruction_str)

{'input': 'I want to calculate the inverse FFT of the data contained in view FFT_SIM_DATA_TBL, where number type of column X1 is imaginary.',
 'output': "The inverse FFT has been calculated and the results are stored in the table 'FFT_SIM_DATA_TBL_FFT_RESULT'."}

In [33]:
cc.table('FFT_SIM_DATA_TBL_FFT_RESULT').collect()

,ID,REAL,IMAG
0,0,0.000000,0.485623
1,1,0.011528,-0.026314
2,2,0.030249,-0.021834
3,3,0.002924,0.001773
4,4,-0.021183,0.042836
...,...,...,...
123,123,0.021655,-0.002071
124,124,0.021183,0.042836
125,125,-0.002924,0.001773
126,126,-0.030249,-0.021834


In [34]:
cc.drop_table("FFT_SIM_DATA_TBL_FFT_RESULT")
cc.drop_table("FFT_SIM_DATA_TBL")
cc.close()